# Máy chủ dịch demo (Qwen3-4B + pair_cuoi)

Không có dữ liệu test.

In [ ]:
# Cấu hình
CAU_HINH = {'gio': 2.0}


In [ ]:
from pathlib import Path
Path('/kaggle/working/may_chu').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/may_chu/dung_mau.py').write_text('"""B3: dựng mẫu huấn luyện (messages) từ hai bộ cụm/pair sau Judge, tách pool và test theo phép chia đã khóa.\n\nMỗi sản phẩm sinh 2 mẫu: title và mo_ta (cho từng nhánh). Mẫu pool -> data/processed/ft_run/mau/pool_{nhanh}.jsonl,\nmẫu test -> data/processed/ft_run/test_khoa/test_{nhanh}.jsonl. Không có gì dẫn xuất từ test (ngoài các số đếm tổng) trong mau/.\nPhép chia chỉ đọc qua chung.doc_split().\n\nChạy:  python -I -X utf8 scripts/ft_train/dung_mau.py --tokenizer <tokenizer.json> [--nhanh cum|pair|ca_hai] [--max-len 2048]\n"""\nimport argparse\nimport json\nimport os\nimport random\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport chung  # noqa: E402\n\n# ---- Chỉ dẫn (B4/zero-shot import lại các hằng số và chi_dan để dùng đúng cùng chữ) ----\nTEN_NGON_NGU = {"zh": "Chinese", "vi": "Vietnamese"}\nCHI_DAN_TITLE = "Translate this product title from {SRC} to {TGT}. Output only the translation.\\n\\n{NGUON}"\n_MO_TA_DAU = "Translate this product\'s attribute list from {SRC} to {TGT}. "\n_MO_TA_CUOI = "\\nProduct title (context only, do not translate): {TITLE}\\n\\n{NGUON}"\nCHI_DAN_MO_TA = {\n    "pair": _MO_TA_DAU + \'Each line is one attribute written as "name: value". \'\n            "Keep the same number and order of lines, and output only the translation." + _MO_TA_CUOI,\n    "cum": _MO_TA_DAU + \'The attributes are separated by "; " and each is written as "name: value". \'\n           \'Keep the same number and order of attributes, separated by "; ", and output only the translation.\' + _MO_TA_CUOI,\n}\n\n# Khuôn chat Qwen3 khi tắt suy luận. CHỈ LÀ XẤP XỈ để đếm token; khuôn thật được áp bằng tokenizer của transformers trên Kaggle.\nKHUON_DEM = "<|im_start|>user\\n{u}<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n{a}<|im_end|>\\n"\nNHANH = ("cum", "pair")\nLOAI = ("title", "mo_ta")\nSEED_MAU = 20261104\nMAX_LEN_MAC_DINH = 2048\n\n\ndef ngon_ngu(huong):\n    """\'zh->vi\' -> (\'Chinese\', \'Vietnamese\')."""\n    s, t = huong.split("->")\n    return TEN_NGON_NGU[s], TEN_NGON_NGU[t]\n\n\ndef chi_dan(loai, nhanh, huong, nguon, title_nguon):\n    """Chỉ dẫn (nội dung user) cho một mẫu. title_nguon chỉ dùng ở mo_ta."""\n    src, tgt = ngon_ngu(huong)\n    if loai == "title":\n        return CHI_DAN_TITLE.format(SRC=src, TGT=tgt, NGUON=nguon)\n    if loai == "mo_ta":\n        return CHI_DAN_MO_TA[nhanh].format(SRC=src, TGT=tgt, TITLE=title_nguon, NGUON=nguon)\n    raise ValueError(f"loai_la:{loai}")\n\n\ndef lam_dem_token(duong_dan_tokenizer):\n    """Trả hàm dem_nhieu([(user, assistant), ...]) -> [số token xấp xỉ theo khuôn Qwen3]."""\n    from tokenizers import Tokenizer\n    tok = Tokenizer.from_file(str(duong_dan_tokenizer))\n\n    def dem_nhieu(cap):\n        if not cap:\n            return []\n        van_ban = [KHUON_DEM.format(u=u, a=a) for u, a in cap]\n        return [len(e.ids) for e in tok.encode_batch(van_ban, add_special_tokens=False)]\n    return dem_nhieu, tok\n\n\ndef _chac(dieu_kien, thong_bao):\n    if not dieu_kien:\n        raise RuntimeError(thong_bao)\n\n\ndef _cat_thuoc_tinh(text, nhanh):\n    return text.split(chung.TACH_DONG[nhanh])\n\n\ndef _dong_mau(r, nhanh, loai, nguon, dich, user, so_token, phan, so_phan, la_test, max_len):\n    pid = str(r.product_id)\n    id_ = f"{pid}_title" if loai == "title" else (f"{pid}_mo_ta" if phan == 0 else f"{pid}_mo_ta_p{phan}")\n    m = {"id": id_, "product_id": pid, "source": r.source, "category": r.category, "huong": r.huong, "loai": loai,\n         "nhanh": nhanh, "fold5": int(r.fold5), "phan": phan, "so_phan": so_phan, "nguon": nguon, "dich": dich,\n         "so_token": int(so_token),\n         "messages": [{"role": "user", "content": user}, {"role": "assistant", "content": dich}]}\n    if la_test:\n        m["qua_dai"] = bool(so_token > max_len)\n    return m\n\n\ndef chia_phan(r, nhanh, nguon, dich, max_len, dem_nhieu):\n    """Chia mô tả pool quá dài thành các phần liên tiếp theo ranh giới thuộc tính (tham lam, mỗi phần <= max_len).\n    Trả (danh_sach_phan, ly_do). danh_sach_phan = [(nguon_phan, dich_phan, user, so_token), ...]; lỗi -> ([], (ly_do, chi_tiet))."""\n    a_n, a_d = _cat_thuoc_tinh(nguon, nhanh), _cat_thuoc_tinh(dich, nhanh)\n    if len(a_n) != len(a_d):\n        return [], ("khong_can_so_thuoc_tinh", f"nguon={len(a_n)} dich={len(a_d)}")\n    tach = chung.TACH_DONG[nhanh]\n\n    def phan_tu(i, k):\n        n, d = tach.join(a_n[i:i + k]), tach.join(a_d[i:i + k])\n        u = chi_dan("mo_ta", nhanh, r.huong, n, r.nguon_title)\n        return n, d, u\n\n    def vua(i, k):\n        n, d, u = phan_tu(i, k)\n        t = dem_nhieu([(u, d)])[0]\n        return t <= max_len, (n, d, u, t)\n\n    ket, i = [], 0\n    while i < len(a_n):\n        ok1, p1 = vua(i, 1)\n        if not ok1:\n            return [], ("thuoc_tinh_don_le_vuot_max_len", f"thuoc_tinh_{i}: {p1[3]} token > {max_len}")\n        lo, hi, tot = 1, len(a_n) - i, p1       # lo luôn là giá trị đã kiểm vừa\n        while lo < hi:\n            mid = (lo + hi + 1) // 2\n            ok, p = vua(i, mid)\n            if ok:\n                lo, tot = mid, p\n            else:\n                hi = mid - 1\n        ket.append(tot)\n        i += lo\n    return ket, None\n\n\ndef dung_mau_nhanh(df, split, nhanh, max_len, dem_nhieu):\n    """Lõi: nhận DataFrame bộ (cột như chung.doc_bo) + split (product_id, la_test, fold5). Không ghi gì ra đĩa.\n    Trả dict: pool, test (list mẫu), loai (list lý do loại, chỉ pool), tong_token_nguyen (pool+test, trước khi chia phần/loại)."""\n    _chac(nhanh in NHANH, f"nhanh_la:{nhanh}")\n    _chac(split.product_id.is_unique and df.product_id.is_unique, "product_id_trung")\n    _chac(set(df.product_id.astype(str)) == set(split.product_id.astype(str)), "tap_sp_bo_va_split_khong_khop")\n    d = df.drop(columns=[c for c in ("la_test", "fold5") if c in df.columns]).copy()\n    d["product_id"] = d.product_id.astype(str)\n    sp = split[["product_id", "la_test", "fold5"]].copy()\n    sp["product_id"] = sp.product_id.astype(str)\n    d = d.merge(sp, on="product_id", how="inner", validate="one_to_one").sort_values("product_id", kind="stable")\n\n    # bản đầy đủ của từng mẫu (chưa chia) và đếm token một lượt\n    ung_vien = []\n    for r in d.itertuples(index=False):\n        ung_vien.append((r, "title", r.nguon_title, r.dich_title,\n                         chi_dan("title", nhanh, r.huong, r.nguon_title, r.nguon_title)))\n        ung_vien.append((r, "mo_ta", r.nguon_mo_ta, r.dich_mo_ta,\n                         chi_dan("mo_ta", nhanh, r.huong, r.nguon_mo_ta, r.nguon_title)))\n    tokens = dem_nhieu([(u, dich) for _, _, _, dich, u in ung_vien])\n\n    pool, test, loai = [], [], []\n    tong_nguyen = {"title": 0, "mo_ta": 0}\n    for (r, lo, nguon, dich, user), t in zip(ung_vien, tokens):\n        tong_nguyen[lo] += t\n        if r.la_test:        # test: giữ nguyên, không chia, không loại; chỉ gắn cờ qua_dai\n            test.append(_dong_mau(r, nhanh, lo, nguon, dich, user, t, 0, 1, True, max_len))\n            continue\n        ghi_loai = lambda ly_do, ct: loai.append({"product_id": str(r.product_id), "loai": lo, "ly_do": ly_do, "chi_tiet": ct})\n        if not str(nguon).strip() or not str(dich).strip():\n            ghi_loai("rong", "nguon hoặc đích rỗng")\n        elif t <= max_len:\n            pool.append(_dong_mau(r, nhanh, lo, nguon, dich, user, t, 0, 1, False, max_len))\n        elif lo == "title":\n            ghi_loai("title_vuot_max_len", f"{t} token > {max_len}")\n        else:\n            cac_phan, ly_do = chia_phan(r, nhanh, nguon, dich, max_len, dem_nhieu)\n            if ly_do:\n                ghi_loai(*ly_do)\n                continue\n            for k, (n, dd, u, tk) in enumerate(cac_phan, 1):\n                pool.append(_dong_mau(r, nhanh, lo, n, dd, u, tk, k, len(cac_phan), False, max_len))\n    return {"pool": pool, "test": test, "loai": loai, "tong_token_nguyen": tong_nguyen}\n\n\ndef kiem_khoa(ket_qua, split, nhanh=""):\n    """Khóa test: dừng (RuntimeError) nếu có bất kỳ vi phạm nào. Gọi TRƯỚC khi ghi file."""\n    pool, test, loai = ket_qua["pool"], ket_qua["test"], ket_qua["loai"]\n    ids_test_split = set(split.product_id[split.la_test].astype(str))\n    ids_pool_split = set(split.product_id[~split.la_test].astype(str))\n    ids_pool = {m["product_id"] for m in pool}\n    ids_test = {m["product_id"] for m in test}\n    chung.kiem_khong_giao(ids_pool, ids_test)                 # không sp nào nằm ở cả hai phía\n    chung.kiem_khong_giao(ids_pool, ids_test_split)           # không mẫu pool nào thuộc test của split\n    _chac(ids_pool <= ids_pool_split, f"{nhanh}: mẫu pool có mã ngoài tập pool của split")\n    _chac(ids_test == ids_test_split, f"{nhanh}: tập mã mẫu test khác tập test của split")\n    _chac(all(0 <= m["fold5"] <= 4 for m in pool), f"{nhanh}: mẫu pool có fold5 ngoài 0..4")\n    _chac(all(m["fold5"] == -1 for m in test), f"{nhanh}: mẫu test có fold5 != -1")\n    # fold của từng mẫu phải đúng với split\n    fold = dict(zip(split.product_id.astype(str), split.fold5))\n    _chac(all(fold[m["product_id"]] == m["fold5"] for m in pool + test), f"{nhanh}: fold5 lệch split")\n    # mọi sp có đủ title + mo_ta (hoặc có lý do loại)\n    co = {(m["product_id"], m["loai"]) for m in pool}\n    bi_loai = {(x["product_id"], x["loai"]) for x in loai}\n    for pid in ids_pool_split:\n        for lo in LOAI:\n            _chac((pid, lo) in co or (pid, lo) in bi_loai, f"{nhanh}: thiếu mẫu {pid}/{lo} (không có lý do loại)")\n    co_t = {(m["product_id"], m["loai"]) for m in test}\n    _chac(co_t == {(p, lo) for p in ids_test_split for lo in LOAI}, f"{nhanh}: tập test thiếu/thừa mẫu")\n    # mỗi mẫu pool chia phần: các phần liên tiếp 1..so_phan\n    theo = {}\n    for m in pool:\n        if m["loai"] == "mo_ta":\n            theo.setdefault(m["product_id"], []).append(m)\n    for pid, ds in theo.items():\n        _chac(sorted(x["phan"] for x in ds) == (list(range(1, len(ds) + 1)) if len(ds) > 1 else [0]),\n              f"{nhanh}: {pid} phần không liên tục")\n        _chac(all(x["so_phan"] == len(ds) for x in ds), f"{nhanh}: {pid} so_phan sai")\n\n\ndef ghi_jsonl(duong_dan, hang):\n    """Ghi tạm rồi đổi tên (atomic)."""\n    duong_dan = Path(duong_dan)\n    duong_dan.parent.mkdir(parents=True, exist_ok=True)\n    tam = duong_dan.with_name(duong_dan.name + ".tmp")\n    with open(tam, "w", encoding="utf-8", newline="\\n") as f:\n        for m in hang:\n            f.write(json.dumps(m, ensure_ascii=False) + "\\n")\n    os.replace(tam, duong_dan)\n\n\ndef _ghi_text(duong_dan, noi_dung):\n    tam = Path(str(duong_dan) + ".tmp")\n    tam.write_text(noi_dung, encoding="utf-8", newline="\\n")\n    os.replace(tam, duong_dan)\n\n\ndef _tong(rows, khoa):\n    out = {}\n    for m in rows:\n        k = str(m[khoa])\n        out[k] = out.get(k, {"so_mau": 0, "so_token": 0})\n        out[k]["so_mau"] += 1\n        out[k]["so_token"] += m["so_token"]\n    return dict(sorted(out.items()))\n\n\ndef lam_bao_cao(ket_qua, nhanh, max_len):\n    """Số liệu cho pool + số đếm mẫu test (chỉ đếm, không nội dung)."""\n    pool, test, loai = ket_qua["pool"], ket_qua["test"], ket_qua["loai"]\n    nguyen = ket_qua["tong_token_nguyen"]\n    return {\n        "nhanh": nhanh, "max_len": max_len,\n        "pool": {"so_mau": len(pool), "so_token": sum(m["so_token"] for m in pool),\n                 "theo_huong": _tong(pool, "huong"), "theo_loai": _tong(pool, "loai"), "theo_fold": _tong(pool, "fold5"),\n                 "so_sp_bi_chia_phan": len({m["product_id"] for m in pool if m["so_phan"] > 1}),\n                 "so_mau_sau_chia_phan": sum(1 for m in pool if m["so_phan"] > 1),\n                 "so_mau_bi_loai": len(loai),\n                 "ly_do_loai": {k: sum(1 for x in loai if x["ly_do"] == k) for k in sorted({x["ly_do"] for x in loai})},\n                 "chi_tiet_loai": loai},\n        "test_chi_dem": {"so_mau": len(test), "so_mau_qua_dai": sum(1 for m in test if m["qua_dai"])},\n        "kiem_token_toan_bo_truoc_chia": {"title": nguyen["title"], "mo_ta": nguyen["mo_ta"],\n                                           "tong": nguyen["title"] + nguyen["mo_ta"],\n                                           "ghi_chu": "pool + test, trước khi chia phần/loại; chỉ là tổng số"},\n    }\n\n\ndef lam_md(ket_qua_theo_nhanh, so_mau=5, seed=SEED_MAU):\n    """5 mẫu POOL ngẫu nhiên (seed cố định) mỗi nhánh x hướng x loại, in đầy đủ messages để đọc tay."""\n    dong = ["# Mẫu pool để đọc tay (B3 dựng mẫu)", "",\n            f"Chọn ngẫu nhiên seed={seed}, {so_mau} mẫu mỗi nhánh x hướng x loại; chỉ lấy từ pool, không có mẫu test.", ""]\n    for nhanh, kq in ket_qua_theo_nhanh.items():\n        for huong in ("zh->vi", "vi->zh"):\n            for lo in LOAI:\n                ds = sorted((m for m in kq["pool"] if m["huong"] == huong and m["loai"] == lo), key=lambda m: m["id"])\n                chon = random.Random(f"{seed}|{nhanh}|{huong}|{lo}").sample(ds, min(so_mau, len(ds)))\n                dong += [f"## nhánh {nhanh} | {huong} | {lo} ({len(ds)} mẫu trong pool)", ""]\n                for m in chon:\n                    dong += [f"### {m[\'id\']}  (source={m[\'source\']}, fold5={m[\'fold5\']}, phan={m[\'phan\']}/{m[\'so_phan\']}, "\n                             f"so_token={m[\'so_token\']})", ""]\n                    for ms in m["messages"]:\n                        dong += [f"**{ms[\'role\']}**", "", "~~~text", ms["content"], "~~~", ""]\n    return "\\n".join(dong) + "\\n"\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser()\n    p.add_argument("--nhanh", default="ca_hai", choices=["cum", "pair", "ca_hai"])\n    p.add_argument("--max-len", type=int, default=MAX_LEN_MAC_DINH)\n    p.add_argument("--tokenizer", required=True, help="đường dẫn tokenizer.json của Qwen3")\n    p.add_argument("--ra-mau", default=str(chung.FT_RUN / "mau"))\n    p.add_argument("--ra-test", default=str(chung.THU_MUC_TEST))\n    a = p.parse_args(argv)\n    ra_mau, ra_test = Path(a.ra_mau), Path(a.ra_test)\n    dem_nhieu, tok = lam_dem_token(a.tokenizer)\n    split = chung.doc_split()                       # khóa test: chỉ đọc phép chia qua đây\n    ds_nhanh = list(NHANH) if a.nhanh == "ca_hai" else [a.nhanh]\n\n    ket, bo = {}, {}\n    for nh in ds_nhanh:\n        bo[nh] = chung.doc_bo(nh)\n        ket[nh] = dung_mau_nhanh(bo[nh], split, nh, a.max_len, dem_nhieu)\n        kiem_khoa(ket[nh], split, nh)\n    # chỉ ghi khi mọi nhánh đã qua kiểm khóa\n    bao_cao = {}\n    for nh in ds_nhanh:\n        ghi_jsonl(ra_mau / f"pool_{nh}.jsonl", ket[nh]["pool"])\n        ghi_jsonl(ra_test / f"test_{nh}.jsonl", ket[nh]["test"])\n        bao_cao[nh] = lam_bao_cao(ket[nh], nh, a.max_len)\n    _ghi_text(ra_mau / "bao_cao_dung_mau.json", json.dumps(bao_cao, ensure_ascii=False, indent=1))\n    _ghi_text(ra_mau / "bao_cao_dung_mau.md", lam_md(ket))\n\n    # so token toàn bộ 25.575 sp với số đo cũ (~17,0M cụm, ~16,3M pair)\n    tham_chieu = {"cum": 17.0e6, "pair": 16.3e6}\n    for nh in ds_nhanh:\n        d = bo[nh]\n        cu = 0\n        for c in ("nguon_title", "dich_title", "nguon_mo_ta", "dich_mo_ta"):\n            cu += sum(len(e.ids) for e in tok.encode_batch(d[c].tolist(), add_special_tokens=False))\n        cu += 40 * 2 * len(d)            # cách đo cũ: token nội dung + ~40 khung mỗi mẫu\n        moi = bao_cao[nh]["kiem_token_toan_bo_truoc_chia"]["tong"]\n        print(f"[{nh}] pool={bao_cao[nh][\'pool\'][\'so_mau\']} mẫu/{bao_cao[nh][\'pool\'][\'so_token\']} token | "\n              f"test={bao_cao[nh][\'test_chi_dem\'][\'so_mau\']} mẫu | loại={bao_cao[nh][\'pool\'][\'so_mau_bi_loai\']} | "\n              f"chia phần={bao_cao[nh][\'pool\'][\'so_sp_bi_chia_phan\']} sp")\n        print(f"[{nh}] tổng token toàn bộ (khuôn+chỉ dẫn, trước chia): {moi:,}; "\n              f"cách đo cũ (nội dung + 40/mẫu): {cu:,}; tham chiếu {tham_chieu[nh]:,.0f}; "\n              f"lệch so tham chiếu {moi / tham_chieu[nh] - 1:+.1%} (mới), {cu / tham_chieu[nh] - 1:+.1%} (đo lại theo cách cũ)")\n        bao_cao[nh]["kiem_token_toan_bo_truoc_chia"].update(\n            {"do_lai_theo_cach_cu": cu, "tham_chieu": tham_chieu[nh]})\n    _ghi_text(ra_mau / "bao_cao_dung_mau.json", json.dumps(bao_cao, ensure_ascii=False, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('/kaggle/working/may_chu/chung.py').write_text('"""Hằng số và hàm dùng chung cho scripts/ft_train (fine-tune Qwen3-4B + LoRA trên hai bộ sau Judge).\n\nQuy tắc khóa tập test (mọi script phải theo):\n  - Chỉ đọc phép chia qua `doc_split()`: hàm này kiểm SHA256 của split.parquet và băm danh sách mã test với\n    DU_LIEU_DONG_BANG.json, lệch là dừng.\n  - Mẫu huấn luyện chỉ lấy từ `la_test == False`; mẫu test ghi riêng vào data/processed/ft_run/test_khoa/.\n  - Không script nào được ghi lại split.parquet ngoài chia_tap.py.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\nGOC = Path(__file__).resolve().parents[2]\nDU_LIEU = GOC / "data" / "processed" / "bo_chon_hf"\nFT_RUN = GOC / "data" / "processed" / "ft_run"\nFILE_SPLIT = FT_RUN / "split.parquet"\nFILE_DONG_BANG = FT_RUN / "DU_LIEU_DONG_BANG.json"\nTHU_MUC_TEST = FT_RUN / "test_khoa"            # mọi thứ dẫn xuất từ tập test (không đưa vào gói huấn luyện)\n\nHF_REPO = "ntquang0410/zh-vie_ecom"\nHF_THU_MUC = "data/LLM/bilingual_judged"\nHF_COMMIT = "82aa9c9"\nFILE_BO = {"cum": "bo_cum_chon_deepseek.parquet", "pair": "bo_pair_chon_deepseek.parquet"}\nFILE_DANH_DAU = {"cum": "bo_cum_danh_dau.parquet", "pair": "bo_pair_danh_dau.parquet"}\nSHA256_HF = {          # lấy từ manifest.json của HF (commit 82aa9c9)\n    "bo_cum_chon_deepseek.parquet": "e0a625b2c1159133af24a8ae43c3ec4af4911ada68e10cf392920b8e49a20def",\n    "bo_pair_chon_deepseek.parquet": "d4535c62ed1e3692f1563143d10c5c612aaec30d5e574eb9969ef003c627230d",\n    "bo_cum_danh_dau.parquet": "945c441f213c0442c788110b836d8aca51f0b36fa7b1e70b87fc2957c38840c6",\n    "bo_pair_danh_dau.parquet": "12b66f24e4bc3718e6c784a927c50c4f0a02c84c1079791bb96734a984bddc24",\n}\nTACH_DONG = {"cum": "; ", "pair": "\\n"}        # ký tự nối các thuộc tính trong mô tả (cả nguồn lẫn đích)\nHUONG = {"1688": "zh->vi", "tiki": "vi->zh"}\nKY_HIEU_XUONG_DONG = " ⏎ "                      # thay "\\n" khi đưa vào CSV gửi Gemini\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for khoi in iter(lambda: f.read(1 << 20), b""):\n            h.update(khoi)\n    return h.hexdigest()\n\n\ndef bam_ds_test(ids):\n    """Băm danh sách mã test (không phụ thuộc thứ tự)."""\n    return hashlib.sha256("\\n".join(sorted(map(str, ids))).encode("utf-8")).hexdigest()\n\n\ndef doc_bo(ten, kiem_hash=True):\n    """Đọc bộ \'cum\' hoặc \'pair\', thêm cột huong, nguon_title, dich_title, nguon_mo_ta, dich_mo_ta."""\n    path = DU_LIEU / FILE_BO[ten]\n    if kiem_hash and sha256_file(path) != SHA256_HF[FILE_BO[ten]]:\n        raise RuntimeError(f"sai_hash_du_lieu:{path}")\n    df = pd.read_parquet(path)\n    z = df.source.eq("1688")\n    df["huong"] = df.source.map(HUONG)\n    df["nguon_title"] = df.title_zh.where(z, df.title_vi)\n    df["dich_title"] = df.title_vi.where(z, df.title_zh)\n    df["nguon_mo_ta"] = df.description_zh.where(z, df.description_vi)\n    df["dich_mo_ta"] = df.description_vi.where(z, df.description_zh)\n    return df\n\n\ndef doc_split():\n    """Đọc phép chia đã khóa. Dừng nếu file bị sửa hoặc danh sách test không khớp bản ghi đóng băng."""\n    if not FILE_SPLIT.exists() or not FILE_DONG_BANG.exists():\n        raise RuntimeError("chua_co_split: chạy scripts/ft_train/chia_tap.py trước")\n    ghi = json.loads(FILE_DONG_BANG.read_text(encoding="utf-8"))["split"]\n    if sha256_file(FILE_SPLIT) != ghi["sha256_file"]:\n        raise RuntimeError("split.parquet_bi_thay_doi")\n    sp = pd.read_parquet(FILE_SPLIT)\n    if bam_ds_test(sp.product_id[sp.la_test]) != ghi["sha256_ds_test"]:\n        raise RuntimeError("ds_test_khong_khop")\n    assert sp.product_id.is_unique\n    assert (sp.fold5[sp.la_test] == -1).all() and sp.fold5[~sp.la_test].between(0, 4).all()\n    return sp\n\n\ndef kiem_khong_giao(ids_huan_luyen, ids_test):\n    """Dừng nếu một mã sản phẩm nằm ở cả hai phía."""\n    giao = set(map(str, ids_huan_luyen)) & set(map(str, ids_test))\n    if giao:\n        raise RuntimeError(f"RO_RI_TEST: {len(giao)} mã test nằm trong dữ liệu huấn luyện, ví dụ {sorted(giao)[:5]}")\n', encoding='utf-8')
Path('/kaggle/working/may_chu/loi_dan.py').write_text('# -*- coding: utf-8 -*-\n"""Dựng lời dẫn đúng chữ lúc huấn luyện (tái dùng dung_mau.chi_dan, không chép lại) và khuôn chat Qwen3 tắt suy luận.\ndung_mau.py và chung.py nằm ở scripts/ft_train trong repo; trên Kaggle/Colab đặt cạnh file này."""\nimport math\nimport sys\nfrom pathlib import Path\n\n_QUA = Path(__file__).resolve()\nfor _p in (_QUA.parent, _QUA.parents[2] / "scripts" / "ft_train"):\n    if (_p / "dung_mau.py").exists():\n        sys.path.insert(0, str(_p))\n        break\nfrom dung_mau import chi_dan  # noqa: E402\n\nKHUON = "<|im_start|>user\\n{u}<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"     # trùng apply_chat_template(enable_thinking=False), đã kiểm\nDUNG = ("<|im_end|>", "<|endoftext|>")\n\n\ndef khuon(u):\n    return KHUON.format(u=u)\n\n\ndef loi_dan_title(zh):\n    return chi_dan("title", "pair", "zh->vi", zh, zh)\n\n\ndef loi_dan_khoi(title_zh, cap):\n    """cap: [(ten, gia_tri), ...] -> khối pair, mỗi thuộc tính một dòng \'tên: giá trị\'."""\n    khoi = "\\n".join(f"{t}: {g}" for t, g in cap)\n    return chi_dan("mo_ta", "pair", "zh->vi", khoi, title_zh), khoi\n\n\ndef chon_tran(nguon):\n    """Trần token mới: 3,5 x độ dài nguồn + 64, tối đa 2048 (như dich_bang_adapter.chon_tran; số ký tự >= số token nên không chặt hơn)."""\n    return min(2048, math.ceil(3.5 * len(nguon)) + 64)\n', encoding='utf-8')
Path('/kaggle/working/may_chu/backend.py').write_text('# -*- coding: utf-8 -*-\n"""Hai cách sinh, cùng giao diện sinh(prompt_chat, tran) -> văn bản, greedy:\n- LlamaServer: llama-server của llama.cpp (laptop; nền GGUF Q5_K_M + --lora).\n- HF: transformers + peft fp16 (Kaggle/Colab T4; đúng cấu hình đã đo ở dòng thứ sáu)."""\nimport json\nimport urllib.request\n\nfrom loi_dan import DUNG\n\n\nclass LlamaServer:\n    def __init__(self, url="http://127.0.0.1:8770", ten="llama-server"):\n        self.url, self.ten = url.rstrip("/"), ten\n\n    def sinh(self, prompt, tran):\n        body = {"prompt": prompt, "n_predict": int(tran), "temperature": 0, "top_k": 1, "seed": 42, "repeat_penalty": 1.0, "cache_prompt": False, "stop": list(DUNG)}\n        req = urllib.request.Request(self.url + "/completion", data=json.dumps(body).encode("utf-8"), headers={"Content-Type": "application/json"})\n        with urllib.request.urlopen(req, timeout=900) as r:\n            return json.loads(r.read().decode("utf-8")).get("content", "")\n\n    def suc_khoe(self):\n        with urllib.request.urlopen(self.url + "/health", timeout=5) as r:\n            return json.loads(r.read().decode("utf-8"))\n\n\nclass HF:\n    def __init__(self, co_so, adapter, ten="hf-fp16", dtype="fp16"):\n        import torch\n        from peft import PeftModel\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        self.torch, self.ten = torch, ten\n        self.tok = AutoTokenizer.from_pretrained(co_so)\n        m = AutoModelForCausalLM.from_pretrained(co_so, dtype={"fp16": torch.float16, "bf16": torch.bfloat16}[dtype], device_map={"": 0}, attn_implementation="sdpa")\n        self.model = PeftModel.from_pretrained(m, adapter).merge_and_unload().eval()       # gộp LoRA vào trọng số: sinh nhanh hơn, cùng kết quả (sai khác làm tròn fp16)\n        self.gpu = torch.cuda.get_device_name(0)\n\n    def sinh(self, prompt, tran):\n        enc = self.tok(prompt, return_tensors="pt", add_special_tokens=False).to(self.model.device)\n        with self.torch.no_grad():\n            out = self.model.generate(**enc, max_new_tokens=int(tran), do_sample=False, pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id)\n        return self.tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)\n\n    def suc_khoe(self):\n        return {"status": "ok", "gpu": self.gpu}\n', encoding='utf-8')
Path('/kaggle/working/may_chu/dich.py').write_text('# -*- coding: utf-8 -*-\n"""Dịch title và khối thuộc tính. Hàm thuần theo một hàm sinh(prompt_chat, tran) nên thử được bằng mô hình giả."""\nimport loi_dan as L\n\nKHOI_TOI_DA = 40       # số dòng tối đa một lượt (extension chia khối ~20 dòng)\n\n\ndef lam_sach(s):\n    return s.strip("\\n").strip()\n\n\ndef dich_title(sinh, zh):\n    zh = zh.strip()\n    if not zh:\n        return ""\n    return lam_sach(sinh(L.khuon(L.loi_dan_title(zh)), L.chon_tran(zh)))\n\n\ndef tach_cap(dong):\n    """\'tên: giá trị\' -> (tên, giá trị) ở dấu \': \' đầu tiên; không có thì (dong, None)."""\n    t, sep, g = dong.partition(": ")\n    return (t, g) if sep else (dong, None)\n\n\ndef dich_khoi(sinh, title_zh, cap):\n    """cap: [(tên, giá trị)] -> (list (tên_vi, giá_trị_vi), thong_ke). Khối ra lệch số dòng thì dịch lại từng dòng (như lúc đo), không đoán dòng."""\n    cap = [(str(t).strip(), str(g).strip()) for t, g in cap if str(t).strip() or str(g).strip()]\n    if not cap:\n        return [], {"so_dong": 0, "lech_dong": False}\n    if len(cap) > KHOI_TOI_DA:\n        raise ValueError(f"khoi_qua_dai:{len(cap)}")\n    u, khoi = L.loi_dan_khoi(title_zh, cap)\n    hyp = lam_sach(sinh(L.khuon(u), L.chon_tran(khoi)))\n    dong = hyp.split("\\n")\n    lech = len(dong) != len(cap)\n    if lech:\n        dong = []\n        for t, g in cap:\n            u1, k1 = L.loi_dan_khoi(title_zh, [(t, g)])\n            dong.append(lam_sach(sinh(L.khuon(u1), L.chon_tran(k1))).split("\\n")[0])\n    return [tach_cap(d) for d in dong], {"so_dong": len(cap), "lech_dong": lech}\n', encoding='utf-8')
Path('/kaggle/working/may_chu/app.py').write_text('# -*- coding: utf-8 -*-\n"""Máy chủ dịch cho extension: POST /dich, GET /suc_khoe. Một lượt sinh tại một thời điểm (khóa), bộ đệm SQLite.\n\n  laptop:  python app.py --llama http://127.0.0.1:8770\n  Kaggle:  python app.py --hf <thư mục nền> <thư mục adapter>      (đặt DEMO_KEY để bật khóa truy cập khi mở ra Internet)\nNghe 127.0.0.1; ra Internet (nếu cần) qua đường hầm. DEMO_KEY đặt thì mọi yêu cầu phải có header X-Demo-Key đúng."""\nimport argparse\nimport hashlib\nimport hmac\nimport json\nimport os\nimport sqlite3\nimport threading\nimport time\n\nfrom flask import Flask, jsonify, request\n\nimport dich as D\nimport loi_dan as L\n\n\nclass BoDem:\n    def __init__(self, duong_dan):\n        self.db = sqlite3.connect(duong_dan, check_same_thread=False)\n        self.db.execute("create table if not exists dem (k text primary key, v text, t real)")\n        self.khoa = threading.Lock()\n\n    def dong(self):\n        self.db.close()\n\n    def lay(self, k):\n        with self.khoa:\n            r = self.db.execute("select v from dem where k=?", (k,)).fetchone()\n        return json.loads(r[0]) if r else None\n\n    def ghi(self, k, v):\n        with self.khoa:\n            self.db.execute("insert or replace into dem values (?,?,?)", (k, json.dumps(v, ensure_ascii=False), time.time()))\n            self.db.commit()\n\n\ndef tao_app(backend, bo_dem, khoa_truy_cap=None):\n    app = Flask(__name__)\n    gpu = threading.Lock()\n\n    def khoa_dem(loai, *phan):\n        return hashlib.sha256(json.dumps([backend.ten, loai, *phan], ensure_ascii=False).encode("utf-8")).hexdigest()\n\n    @app.before_request\n    def kiem_khoa():\n        if khoa_truy_cap and request.method != "OPTIONS" and not hmac.compare_digest(request.headers.get("X-Demo-Key", ""), khoa_truy_cap):\n            return jsonify({"loi": "khong_co_quyen"}), 401\n\n    @app.after_request\n    def cors(r):\n        r.headers["Access-Control-Allow-Origin"] = "*"\n        r.headers["Access-Control-Allow-Headers"] = "Content-Type, X-Demo-Key"\n        return r\n\n    @app.route("/suc_khoe")\n    def suc_khoe():\n        try:\n            return jsonify({"mo_hinh": backend.ten, **backend.suc_khoe()})\n        except Exception as e:      # noqa: BLE001\n            return jsonify({"loi": f"{type(e).__name__}: {e}"}), 503\n\n    @app.route("/dich", methods=["POST", "OPTIONS"])\n    def dich():\n        if request.method == "OPTIONS":\n            return ("", 204)\n        j = request.get_json(force=True)\n        loai = j.get("loai")\n        try:\n            if loai == "title":\n                k = khoa_dem("title", j["zh"])\n                v = bo_dem.lay(k)\n                if v is None:\n                    t0 = time.time()\n                    with gpu:\n                        v = {"vi": D.dich_title(backend.sinh, j["zh"]), "giay": round(time.time() - t0, 2)}\n                    bo_dem.ghi(k, v)\n                    v["dem"] = False\n                else:\n                    v["dem"] = True\n                return jsonify(v)\n            if loai == "khoi":\n                cap = [(t, g) for t, g in j["cap"]]\n                k = khoa_dem("khoi", j["title"], cap)\n                v = bo_dem.lay(k)\n                if v is None:\n                    t0 = time.time()\n                    with gpu:\n                        ra, tk = D.dich_khoi(backend.sinh, j["title"], cap)\n                    v = {"cap": ra, **tk, "giay": round(time.time() - t0, 2)}\n                    bo_dem.ghi(k, v)\n                    v["dem"] = False\n                else:\n                    v["dem"] = True\n                return jsonify(v)\n        except (KeyError, ValueError) as e:\n            return jsonify({"loi": f"yeu_cau_sai:{e}"}), 400\n        return jsonify({"loi": "loai_la"}), 400\n\n    return app\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    g = ap.add_mutually_exclusive_group(required=True)\n    g.add_argument("--llama")\n    g.add_argument("--hf", nargs=2, metavar=("CO_SO", "ADAPTER"))\n    ap.add_argument("--cong", type=int, default=8765)\n    ap.add_argument("--dem", default="bo_dem.sqlite")\n    ap.add_argument("--ten", default=None)\n    a = ap.parse_args()\n    from backend import HF, LlamaServer\n    backend = LlamaServer(a.llama, a.ten or "llama-q5km-pair_cuoi") if a.llama else HF(a.hf[0], a.hf[1], a.ten or "hf-fp16-pair_cuoi")\n    tao_app(backend, BoDem(a.dem), os.environ.get("DEMO_KEY")).run(host="127.0.0.1", port=a.cong, threaded=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('/kaggle/working/may_chu/kaggle_may_chu.py').write_text('# -*- coding: utf-8 -*-\n"""Chạy máy chủ dịch trên Kaggle (GPU T4, Internet bật) và mở ra qua đường hầm Cloudflare để extension gọi.\nDùng đúng cấu hình đã đo ở dòng thứ sáu (Qwen3-4B fp16 + LoRA), adapter pair_cuoi (demo), một GPU (GPU thứ hai không cần).\nIn ra log các dòng \'KET_NOI url=... key=...\' để đọc bằng `kaggle kernels logs`; khóa truy cập sinh ngẫu nhiên mỗi phiên.\nHết giờ (cfg[\'gio\']) thì tự tắt."""\nimport json\nimport os\nimport re\nimport secrets\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nGOC = Path("/kaggle/working/may_chu")\nCLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"\n\n\ndef sh(c, **kw):\n    return subprocess.run(c, shell=True, text=True, capture_output=True, **kw)\n\n\ndef goi_json(url, body, khoa, timeout=300):\n    req = urllib.request.Request(url, data=json.dumps(body).encode("utf-8"), headers={"Content-Type": "application/json", "X-Demo-Key": khoa})\n    t0 = time.time()\n    with urllib.request.urlopen(req, timeout=timeout) as r:\n        return json.loads(r.read().decode("utf-8")), time.time() - t0\n\n\ndef main(cfg):\n    r = sh("pip install -q \'peft==0.18.1\' flask 2>&1 | tail -2")\n    print("pip:", r.stdout.strip()[-300:], flush=True)\n    sys.path.insert(0, str(GOC))\n    os.chdir(GOC)\n    import torch\n    print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpu", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], flush=True)\n    co_so = next(Path("/kaggle/input").rglob("REVISION.txt")).parent\n    adapter = next(p for p in Path("/kaggle/input").rglob("pair_cuoi/ep2/adapter_model.safetensors")).parent\n    print("co_so", co_so, "| revision", co_so.joinpath("REVISION.txt").read_text().split(), "| adapter", adapter, flush=True)\n\n    from backend import HF\n    import app as A\n    t0 = time.time()\n    backend = HF(str(co_so), str(adapter), ten=f"kaggle-fp16-pair_cuoi")\n    print(f"nạp mô hình {time.time() - t0:.0f}s trên {backend.gpu}", flush=True)\n    khoa = secrets.token_urlsafe(24)\n    app = A.tao_app(backend, A.BoDem(str(GOC / "bo_dem.sqlite")), khoa)\n    from werkzeug.serving import make_server\n    srv = make_server("127.0.0.1", 8765, app, threaded=True)\n    threading.Thread(target=srv.serve_forever, daemon=True).start()\n\n    sh(f"curl -fsSL -o cloudflared {CLOUDFLARED} && chmod +x cloudflared")\n    cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(GOC))\n    url, dong = None, []\n    t0 = time.time()\n    while url is None and time.time() - t0 < 120:\n        l = cf.stdout.readline()\n        if not l:\n            break\n        dong.append(l.strip())\n        m = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", l)\n        if m:\n            url = m.group(0)\n    threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()       # xả đệm để cloudflared không nghẽn\n    if not url:\n        print("KHONG_MO_DUOC_DUONG_HAM", dong[-8:], flush=True)\n        return\n    print(f"KET_NOI url={url} key={khoa}", flush=True)\n\n    # tự kiểm qua đường hầm công khai (như extension sẽ gọi)\n    time.sleep(8)\n    kq = {"url": url, "gpu": backend.gpu}\n    try:\n        kq["title"], kq["giay_title"] = goi_json(url + "/dich", {"loai": "title", "zh": "源头厂家高级感立体短袖T恤男夏季薄款新款潮牌圆领半袖男士体恤"}, khoa)\n        cap = [["品牌", "Elmich"], ["材质", "棉麻"], ["颜色", "黑色"], ["是否含遥控", "否"], ["容量", "500ml"]] * 4\n        kq["khoi20"], kq["giay_khoi20"] = goi_json(url + "/dich", {"loai": "khoi", "title": "Elmich 不锈钢保温杯 500ml 黑色", "cap": cap}, khoa)\n        kq["khoi20_lech_dong"] = kq["khoi20"].get("lech_dong")\n        kq["khong_khoa"] = None\n        try:\n            urllib.request.urlopen(urllib.request.Request(url + "/dich", data=b"{}", headers={"Content-Type": "application/json"}), timeout=30)\n        except urllib.error.HTTPError as e:\n            kq["khong_khoa"] = e.code\n    except Exception as e:      # noqa: BLE001\n        kq["loi_tu_kiem"] = f"{type(e).__name__}: {e}"\n    print("TU_KIEM", json.dumps(kq, ensure_ascii=False), flush=True)\n\n    het = time.time() + cfg.get("gio", 2.0) * 3600\n    while time.time() < het:\n        time.sleep(300)\n        print(f"song {int((het - time.time()) / 60)} phut nua", flush=True)\n    print("HET_GIO", flush=True)\n', encoding='utf-8')


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/may_chu')
import kaggle_may_chu
kaggle_may_chu.main(CAU_HINH)
